# cartpole-term: fresh runtime -- train + predictor (Part A), then out-of-distribution generalization test (Parts B-D)

Fresh Colab runtime, no prior state. This shim installs deps, unpacks the
uploaded code, and shells out to `scripts/*.py` only -- no pipeline logic in
cells (CLAUDE.md S4.2).

**Part A** reproduces the gated pipeline for seed 0: train -> select
checkpoints -> collect 3 frozen datasets -> train+evaluate the predictor,
exactly as validated in the completed run (README.md). Its output,
`outputs/predictor_target_seed0.pt`, is a fully-trained, frozen predictor.

**Parts B-D are new, outside CLAUDE.md's gated spec** (see DEVIATIONS.md,
"Phase 2"): train a **second, completely independent** policy (new seed, own
random trajectory), then test the seed-0 predictor -- frozen, never
retrained, never influencing this second run in any way -- against **every
checkpoint that second run saves**, purely at inference time. This answers a
real out-of-distribution question the original 12 gates don't: does a
predictor trained on one policy's `target`-competence data generalize to an
entirely different policy it has never seen?


## 1. Install deps + confirm GPU

In [ ]:
!pip install -q mjlab==1.6.0 rsl-rl-lib==5.4.2 scikit-learn "imageio[ffmpeg]" pillow
import torch
assert torch.cuda.is_available(), "No GPU visible -- check Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))


## 2. Upload and unpack the code

When prompted, pick `cartpole_term_code.zip` from the repo root (build it with `python3 tools/build_code_zip.py`)
on your local disk. It must include `scripts/eval_predictor_generalization.py`,
`scripts/plot_predictor_generalization.py`, and `scripts/train_stratified_predictor.py`
alongside the usual `src/`, `scripts/`, `pyproject.toml`.


In [ ]:
from google.colab import files
import zipfile, re, os

uploaded = files.upload()
name = list(uploaded.keys())[0]
canonical = re.sub(r" \(\d+\)(?=\.\w+$)", "", name)  # strip Colab's " (1)" collision suffix
if canonical != name:
    os.rename(name, canonical)

os.makedirs("/content/cartpole-term", exist_ok=True)
with zipfile.ZipFile(canonical) as zf:
    zf.extractall("/content/cartpole-term")

%cd /content/cartpole-term
!ls scripts/eval_predictor_generalization.py scripts/plot_predictor_generalization.py scripts/train_stratified_predictor.py

---
## Part A: seed 0 pipeline (train -> select -> collect x3 -> predictor)

### 3. Train seed 0 (dense eval grid + early stop)

In [ ]:
!python scripts/train.py --seeds 0 --eval.interval 2 --early-stop-patience 2


### 4. Select checkpoints at the three operating points

In [ ]:
!python scripts/select_checkpoints.py --seeds 0


### 5. Collect frozen datasets (one operating point per cell)

In [ ]:
!python scripts/collect_frozen.py --seeds 0 --operating-point struggling


In [ ]:
!python scripts/collect_frozen.py --seeds 0 --operating-point target


In [ ]:
!python scripts/collect_frozen.py --seeds 0 --operating-point competent


### 6. Train + evaluate the predictor

`--operating-point target` is the one Parts B-D will reuse
(`outputs/predictor_target_seed0.pt`) -- run it explicitly first in case you
want to skip struggling/competent on a time-limited runtime; all three are
still included below for a complete Part A.


In [ ]:
!python scripts/eval_predictor.py --seeds 0 --operating-point target
!python scripts/eval_predictor.py --seeds 0 --operating-point struggling
!python scripts/eval_predictor.py --seeds 0 --operating-point competent


### 7. Download Part A outputs (safety checkpoint before Part B)

In [ ]:
import shutil
shutil.make_archive("/content/outputs_partA", "zip", "/content/cartpole-term/outputs")
files.download("/content/outputs_partA.zip")


---
## Part B: a second, independent training run

New seed (3 -- distinct from the 0/1/2 already used), same training
invocation shape as Part A so the checkpoint cadence lines up with
`eval_history.json` the same way (`--eval.interval 2` divides evenly into
the default `save_interval=10`, so every saved `model_*.pt` has a matching
frozen-eval measurement for Part D's plot). Nothing about this run is
special-cased for the predictor -- it's an ordinary PPO run.


In [ ]:
!python scripts/train.py --seeds 3 --eval.interval 2 --early-stop-patience 2


### 8. Find this run's log directory (timestamped, so glob for it)

In [ ]:
import glob

candidates = sorted(glob.glob("logs/rsl_rl/cartpole_term/*_seed3"))
assert candidates, "no seed-3 run directory found -- did the training cell above finish?"
run_dir = candidates[-1]  # most recent, in case of a stale prior attempt
print("run_dir =", run_dir)
print("checkpoints:", sorted(glob.glob(run_dir + "/model_*.pt")))


---
## Part C: heavy out-of-distribution eval

Loads the ALREADY-TRAINED `outputs/predictor_target_seed0.pt` (from Part A,
step 6) and runs it -- inference only, no retraining, no influence on this
second run whatsoever -- against every checkpoint found above. Each
checkpoint gets its own full-size rollout (1M labelled transitions / 1,000
terminated episodes / 4096 envs, same targets `collect_frozen.py` uses for
the gated datasets -- collection is cheap, ~15-20s per 1M transitions, so
there's no reason to cut corners across ~10-20 checkpoints). This can take a
while depending on how many checkpoints seed 3 produced; each one prints its
own line as it finishes.


In [ ]:
!python scripts/eval_predictor_generalization.py \
    --predictor-weights outputs/predictor_target_seed0.pt \
    --log-dir {run_dir}


---
## Part D: graph predictor quality vs. this second run's own competence

Joins Part C's per-checkpoint metrics against seed 3's own
`eval_history.json` (its frozen in-training eval, CLAUDE.md S6) on
iteration, so the x-axis is the second run's *own measured* policy
competence -- not iteration count, and not anything derived from the
predictor's original training run. Produces one PNG with four panels
(competence curve, AUPRC vs. competence, AUROC vs. competence, detection
lead time vs. competence) plus a JSON of the joined table, and prints the
same table to the cell output.


In [ ]:
import glob

gen_json = sorted(glob.glob("outputs/predictor_generalization_predictor_target_seed0_*.json"))[-1]
print("generalization json:", gen_json)

!python scripts/plot_predictor_generalization.py \
    --generalization-json {gen_json} \
    --eval-history {run_dir}/eval_history.json \
    --out outputs/predictor_generalization_seed3.png


In [ ]:
from IPython.display import Image, display
display(Image("outputs/predictor_generalization_seed3.png"))


### 9. Download Part B-D outputs

In [ ]:
import shutil
shutil.make_archive("/content/outputs_generalization", "zip", "/content/cartpole-term/outputs")
files.download("/content/outputs_generalization.zip")

shutil.make_archive("/content/logs_seed3", "zip", run_dir)
files.download("/content/logs_seed3.zip")


---
## Part E: one predictor across (most of) the competence range

The seed-3 sweep above showed the `target`-trained predictor generalizes
well from competence ~70 through ~380 steps, then breaks down sharply once
a checkpoint saturates the 1000-step ceiling (base rate under 1%, an order
of magnitude more extreme than `competent`'s own 4.5%). This trains ONE
predictor on the union of `struggling` + `target` + `competent` (already
collected in Part A) plus a freshly-collected `near_ceiling` dataset from
seed 0's own latest checkpoint -- same architecture/loss/optimizer as
everywhere else (CLAUDE.md S8.2/S8.3), just fed a training distribution
that actually covers the regime that broke. Reports a per-source test
breakdown so it's visible whether this actually helps every regime or just
dilutes toward the middle.


### 10. Find seed 0's own latest (most-converged) checkpoint

In [ ]:
import glob, re

seed0_dirs = sorted(glob.glob("logs/rsl_rl/cartpole_term/*_seed0"))
assert seed0_dirs, "no seed-0 run directory found -- did Part A's training cell finish?"
seed0_dir = seed0_dirs[-1]

ckpts = glob.glob(seed0_dir + "/model_*.pt")
latest_ckpt = max(ckpts, key=lambda p: int(re.search(r"model_(\d+)\.pt$", p).group(1)))
print("seed0_dir =", seed0_dir)
print("latest_ckpt =", latest_ckpt)


### 11. Collect the near-ceiling dataset and train the unified predictor

In [ ]:
!python scripts/train_unified_predictor.py --seed 0 \
    --near-ceiling-checkpoint {latest_ckpt}


### 12. Re-run the generalization sweep with the unified predictor

In [ ]:
!python scripts/eval_predictor_generalization.py \
    --predictor-weights outputs/predictor_unified_seed0.pt \
    --log-dir {run_dir}


### 13. Plot it against the original target-only predictor's sweep

In [ ]:
import glob

gen_json_unified = sorted(glob.glob("outputs/predictor_generalization_predictor_unified_seed0_*.json"))[-1]
print("unified generalization json:", gen_json_unified)

!python scripts/plot_predictor_generalization.py \
    --generalization-json {gen_json_unified} \
    --eval-history {run_dir}/eval_history.json \
    --out outputs/predictor_generalization_unified_seed3.png


In [ ]:
from IPython.display import Image, display
print("target-only predictor (Part D):")
display(Image("outputs/predictor_generalization_seed3.png"))
print("unified predictor (Part E):")
display(Image("outputs/predictor_generalization_unified_seed3.png"))


### 14. Download Part E outputs

In [ ]:
import shutil
shutil.make_archive("/content/outputs_unified", "zip", "/content/cartpole-term/outputs")
files.download("/content/outputs_unified.zip")


---
## Part F: 4 equal-size competence stacks, collected in parallel, trained with stratified sampling

Ground-up rewrite, replacing the earlier multi-checkpoint attempt. That
attempt's own per-source breakdown showed why it didn't work: the
near-ceiling source ended up with 5x more test data than every other
source (collect_one keeps running until it hits a terminated-EPISODE
target, and a low-base-rate policy takes far longer to accumulate enough
failures) -- and it still failed. So it was never under-represented by
volume; pooling by natural proportions and sampling one shuffled DataLoader
just isn't enough on its own.

This version fixes it structurally: 4 named checkpoints (struggling,
target, competent, fluent -- the last averaging ~1000 steps, auto-selected
from the run's own eval_history.json) are collected SIMULTANEOUSLY, one
env per checkpoint stepped round-robin, each stopped at the SAME simple
transitions threshold (no terminated-episode target at all, so every stack
ends up the same size by construction). Training draws each minibatch by
sampling an equal number of rows from each of the 4 stacks, and early
stopping tracks the MEAN of the 4 stacks' own validation AUPRC -- not one
pooled number a large easy stack can dominate.


### 15. Train the stratified predictor

No extra checkpoint-discovery cells needed this time -- struggling/target/
competent come straight from `outputs/selected_checkpoints.json` (Part A,
step 4) and `fluent` is auto-selected inside the script from seed 0's own
`eval_history.json`.


In [ ]:
!python scripts/train_stratified_predictor.py --seed 0

### 16. Re-run the generalization sweep with the stratified predictor

`--predictor-hidden 128,128` matches this script's default width (reverted
from 256,256 -- the whole point of equal-size stacks is a much smaller
total dataset than the previous attempt pooled).


In [ ]:
!python scripts/eval_predictor_generalization.py \
    --predictor-weights outputs/predictor_stratified_seed0.pt \
    --predictor-hidden 128,128 \
    --log-dir {run_dir}

### 17. Plot against the earlier predictors' sweeps


In [ ]:
import glob

gen_json_stratified = sorted(glob.glob("outputs/predictor_generalization_predictor_stratified_seed0_*.json"))[-1]
print("stratified generalization json:", gen_json_stratified)

!python scripts/plot_predictor_generalization.py \
    --generalization-json {gen_json_stratified} \
    --eval-history {run_dir}/eval_history.json \
    --out outputs/predictor_generalization_stratified_seed3.png

In [ ]:
from IPython.display import Image, display
print("target-only predictor (Part D):")
display(Image("outputs/predictor_generalization_seed3.png"))
print("unified predictor (Part E):")
display(Image("outputs/predictor_generalization_unified_seed3.png"))
print("stratified predictor (Part F):")
display(Image("outputs/predictor_generalization_stratified_seed3.png"))

### 18. Download Part F outputs


In [ ]:
import shutil
shutil.make_archive("/content/outputs_stratified", "zip", "/content/cartpole-term/outputs")
files.download("/content/outputs_stratified.zip")

---
## Part G: video of the stratified predictor across all 4 checkpoints

One mp4, four segments back to back (struggling -> target -> competent ->
fluent), same frozen stratified predictor overlaid on each -- probability
bar, threshold marker, and ground-truth label once known, same as the
gated predictor_overlay.mp4 but across all 4 competence regimes instead of
just `target`. Each segment runs until 2 failures are captured or 20s
elapse, whichever's first -- `fluent` will likely just hit the 20s cap
with few or no failures, which is itself the honest result for that
regime (see DEVIATIONS.md).


### 19. Record the video


In [ ]:
!python scripts/record_predictor_stratified_video.py --seed 0

### 20. Preview + download


In [ ]:
from IPython.display import Video, display
display(Video("outputs/predictor_stratified_overlay_seed0.mp4", embed=True, width=640))

In [ ]:
files.download("outputs/predictor_stratified_overlay_seed0.mp4")